# 02 — Exploratory Data Analysis

Loads `df_development` from `data/processed/` (produced by `01_data_loading_cleaning.ipynb`) and covers:

- Suicide rate evolution by country and by EU region
- Cross-country distribution
- Feature distributions and skewness
- Outlier detection (IQR)
- Multicollinearity check (VIF) and the resulting drop of `Eating disorders`

Feature engineering (lags) and modelling are covered in later notebooks.

In [ ]:
import sys

sys.path.append("..")

import pandas as pd

from src import (
    ID_COLS,
    TARGET,
    EU_REGIONS,
    SOCIAL_ECONOMIC_FEATURES,
    compute_vif,
    flag_outliers_iqr,
    build_predictor_list,
    suicide_evolution_graph,
    plot_suicide_trend_by_region,
    plot_suicide_boxplot_by_country,
    plot_feature_distributions,
    plot_suicide_dispersion_stripplot,
    plot_vif_bar,
    save_figure,
)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)

df_development = pd.read_parquet("../data/processed/df_development.parquet")
print(
    f"df_development: {df_development.shape[0]} rows, {df_development.shape[1]} columns"
)
display(df_development.head())

fig_prefix = "02_"

## Suicide rate evolution — country spotlights

Highest and lowest average suicide rate in the EU (Lithuania and Greece), plus Germany as a mid-range reference. This gives context for the outliers that are formally flagged later in the notebook — a country sitting far from the EU average is not automatically a data error.

In [ ]:
fig = suicide_evolution_graph(df_development, "LTU", "Lithuania")  # highest average
path = save_figure(fig, name="suicide_evolution_ltu", prefix=fig_prefix)
fig = suicide_evolution_graph(df_development, "GRC", "Greece")  # lowest average
path = save_figure(fig, name="suicide_evolution_grc", prefix=fig_prefix)
fig = suicide_evolution_graph(df_development, "DEU", "Germany")  # mid-range reference
path = save_figure(fig, name="suicide_evolution_deu", prefix=fig_prefix)

## Suicide rate trends by EU region (2000-2021)

Countries are grouped into four regions (`EU_REGIONS` in `src/config.py`: Western Europe/Nordics, Mediterranean, Eastern Europe, Baltics) based on geographic and cultural proximity rather than any statistical clustering. The idea is to check whether suicide rate trends move together within these informal groupings — which would suggest shared structural factors (history, culture, healthcare system design) beyond what the individual socioeconomic indicators capture.

In [ ]:
df_development["Region"] = df_development["Code"].map(EU_REGIONS)
fig = plot_suicide_trend_by_region(df_development)
path = save_figure(fig, name="suicide_trend_by_region", prefix=fig_prefix)

## Cross-country distribution analysis

Boxplot of the full suicide rate distribution per country (2000–2021), ordered by median. This complements the three country spotlights above by showing the spread and overlap across *all* 27 countries at once, rather than just three examples.

In [ ]:
fig = plot_suicide_boxplot_by_country(df_development)
path = save_figure(fig, name="suicide_boxplot_by_country", prefix=fig_prefix)

## Feature distribution analysis

Histograms + KDE for all numerical predictors, annotated with skewness. Beyond describing the data, this matters for two decisions made later: heavily skewed features can distort linear models (Ridge/Lasso) more than tree-based ones, and skew driven by extreme values is part of the motivation for using `RobustScaler` (median/IQR-based) instead of a mean-based `StandardScaler` when scaling for modelling in `04_models.ipynb`.

In [ ]:
predictor_features = build_predictor_list(df_development, ID_COLS, TARGET)

fig = plot_feature_distributions(df_development, predictor_features)
path = save_figure(fig, name="feature_distributions", prefix=fig_prefix)

print("\nSkewness summary:")
print(df_development[predictor_features].skew().sort_values(ascending=False).round(2))

## Outlier detection (IQR method)

Flags values falling outside `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]` for each socioeconomic feature independently. These outliers are **flagged for awareness, not removed** — some of them are genuine (e.g. a country with an unusually high Gini index), and removing them would throw away real signal. This is also why `RobustScaler`, rather than outlier removal, is the chosen way of handling them during modelling.

In [ ]:
socioeconomic_cols_for_outliers = [
    c for c in SOCIAL_ECONOMIC_FEATURES if c in df_development.columns
]

outlier_summary = flag_outliers_iqr(df_development, socioeconomic_cols_for_outliers)
print("Outlier Summary (IQR method, threshold=1.5):")
display(outlier_summary)

fig = plot_suicide_dispersion_stripplot(df_development)
path = save_figure(fig, name="suicide_dispersion_stripplot", prefix=fig_prefix)

## Multicollinearity check (VIF)

The Variance Inflation Factor quantifies how much the variance of a feature's estimated coefficient is inflated because that feature is linearly predictable from the other predictors. VIF > 5: moderate concern; VIF > 10: high concern — at that point the feature is carrying mostly redundant information rather than independent signal, which destabilises linear model coefficients.

In [ ]:
vif_results = compute_vif(df_development, predictor_features)
print("Variance Inflation Factor (VIF) — full predictor set:")
display(vif_results)
fig = plot_vif_bar(vif_results)
path = save_figure(fig, name="vif_bar", prefix=fig_prefix)

### Addressing high multicollinearity: dropping `Eating disorders`

`Eating disorders` has a VIF of 12.098 (see `src/config.py`) — high enough that it shares most of its variance with the other mental-health predictors and adds almost no independent information to the model. It is dropped before modelling, and VIF is recomputed on the reduced predictor set to confirm the remaining features are within acceptable bounds.

In [ ]:
df_development = df_development.drop(columns=["Eating disorders"], errors="ignore")
predictor_features = build_predictor_list(df_development, ID_COLS, TARGET)

vif_results = compute_vif(df_development, predictor_features)
print("Variance Inflation Factor (VIF) — after dropping Eating disorders:")
display(vif_results)
fig = plot_vif_bar(vif_results)
path = save_figure(fig, name="vif_bar", prefix=fig_prefix)

## Save EDA-cleaned dataset

Overwrites `df_development.parquet` with `Region` added and `Eating disorders` dropped, so downstream notebooks pick up both changes.

In [ ]:
df_development.to_parquet("../data/processed/df_development.parquet", index=False)
print("Saved: data/processed/df_development.parquet (updated)")